# Silver Layer — Reviews Transformation

Transforms `bronze.order_reviews` into `silver.order_reviews`. Deduplicates rows using a window function (latest answer wins per review_id), and trims free-text comment fields.

Rows with null `order_id`, null `review_score`, null `review_creation_date`, or a `review_score` outside 1–5 are quarantined to `quarantine.order_reviews` with a `violation_reasons` column. Clean rows go to Silver.

## Design Decisions

* **Window function deduplication, latest-answer-wins** — Source data has duplicate `review_id` values. Rather than arbitrary dedup, we use `row_number()` over `review_id` ordered by `review_answer_timestamp` descending. This treats the most recent answer as the canonical version of the review.
* **Quarantine over filter** — 8,855 rows fail one of the validity rules. Quarantining preserves them with violation reasoning for investigation rather than silently dropping.
* **Trim-only for comment fields** — Comment titles and messages are natural language. We trim whitespace but do not lower-case, since case carries meaning in free text.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
quarantine_schema = 'quarantine'

### Transformations

In [0]:
df_reviews = spark.read.table(f"{catalog_name}.{bronze_schema}.order_reviews")
window = Window.partitionBy('review_id').orderBy(col('review_answer_timestamp').desc())
transformed_reviews_df = df_reviews.filter(col('review_id').isNotNull()).withColumn('rn', row_number().over(window)).filter(col('rn')==1).drop('rn')\
    .withColumn('review_comment_title',trim(col('review_comment_title')))\
        .withColumn('review_comment_message',trim(col('review_comment_message')))\
            .withColumn('silver_processed_at', current_timestamp())


### Identify invalid order_id, review_score and review_creation_date
Validates order_id, review_score and review_creation_date columns which have nulls and review_score outside 1-5.
Split the invalid and clean records into different dataframe by using filtering and add violation_reasons column in quarantine dataframe.

In [0]:
quarantine_condition = (col('order_id').isNull())  | (col('review_score').isNull()) | (col('review_score') < 1) | (col('review_score') > 5) | (col('review_creation_date').isNull())

df_quarantine_reviews = transformed_reviews_df.filter(quarantine_condition).withColumn('violation_reasons',
    concat_ws('; ',
        when(col('order_id').isNull(), lit('order_id is null')),
        when(col('review_score').isNull(), lit('review_score is null')),
        when(col('review_score') < 1, lit('review_score < 1')),
        when(col('review_score') > 5, lit('review_score > 5')),
        when(col('review_creation_date').isNull(), lit('review_creation_date is null'))
    )
)

df_clean_reviews = transformed_reviews_df.filter(~quarantine_condition)

## Writing quarantine dataframe to quarantine schema

In [0]:
df_quarantine_reviews.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{quarantine_schema}.order_reviews')


### Data Quality Checks

In [0]:
check_not_null(df_clean_reviews, ['review_id','order_id','review_score','review_creation_date'])
check_unique(df_clean_reviews,['review_id'])
check_row_count(df_clean_reviews,f'{catalog_name}.{bronze_schema}.order_reviews',85,100)

check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED


## Writing clean dataframe to silver schema

In [0]:
df_clean_reviews.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.order_reviews')